This notebook creates fixed sized portfolios - (approach changed based on initial testing results)

In [1]:
import pandas as pd
import numpy as np
import time
import datetime
import os
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler


In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
max_portfolio_size = 50 # maximum number of items a portfolio is allowed to have
min_portfolio_size = 20 # minimum number of items a portfolio is allowed to have
user_id_item_size = 10 # number of items that is cosidered to create the user_id as a sequence of items
test_items_size = 5 # numbbre of items that are left as the test portion of the portfolio
minimum_item_occurance = 10

# Utils

In [4]:
def split_symbol(symbol):
    return symbol.split('.')[0]

In [5]:
def mapper(symb, mapper):
    try:
        return mapper.get(symb)
    except:
        return 'Empty'

In [6]:
def get_max_values(df, col_name = 'RATING', round_method = 'round'):
    max_rating_row = df.loc[df[col_name].idxmax()]
    if round_method == 'round':
        max_rating_row[col_name] = max_rating_row[col_name].round(0)
        
    elif round_method == 'ceil':
        max_rating_row[col_name] = np.ceil(max_rating_row[col_name].array)
    
    return max_rating_row

In [7]:
def infer_rating(df, qnt_col = 'SHARESQTY', price_col= 'SHAREPRICE', rating_col = 'RATING'):

    order_prices = df[qnt_col] * df[price_col]
    scaler = MinMaxScaler(feature_range=(1,5))
    scaled_price = scaler.fit_transform(order_prices.values.reshape(-1, 1))
    df[rating_col] = np.clip(scaled_price, 1, 5)

    df = df.groupby("STOCKCODE", group_keys = False).apply(lambda x: get_max_values(x)).reset_index(drop = True)
    return df

In [8]:
def to_timestamp(date):
    return datetime.datetime.timestamp(date)

In [9]:
def create_user_portfolios(df, time_col = 'UNIX_TS', latest_k = max_portfolio_size, user_id_item_size = user_id_item_size, test_item_size = test_items_size):
    df = df.sort_values(time_col, ascending = True)
    df = df.tail(latest_k)
    df = df.reset_index(drop = True)
    user_id_as_items = df.iloc[-(test_items_size+user_id_item_size) : -test_items_size].STOCKCODE.values
    # df['Test_'] = df.iloc[-test_items_size:]
    df['test_'] = 0
    df.loc[len(df)-test_items_size:, 'test_'] = 1
    df['USER_ID'] = (np.repeat(user_id_as_items.reshape(1,-1), len(df), axis = 0)).tolist()

    return df


# Data reading


In [10]:
data_path = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2\raw"

portfolios = pd.DataFrame()
for fname in os.listdir(data_path):
    
    broker_df_ = pd.read_csv(os.path.join(data_path,fname), sep = '|')
    print("--- reading : {}".format(fname))
    
    portfolios = pd.concat([portfolios, broker_df_], ignore_index = True)

C:\Users\naradaw\AppData\Local\Temp\ipykernel_13888\480125734.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  broker_df_ = pd.read_csv(os.path.join(data_path,fname), sep = '|')


--- reading : Bartleet.txt


C:\Users\naradaw\AppData\Local\Temp\ipykernel_13888\480125734.py:6: DtypeWarning: Columns (2,7) have mixed types. Specify dtype option on import or set low_memory=False.
  broker_df_ = pd.read_csv(os.path.join(data_path,fname), sep = '|')


--- reading : CAS.txt
--- reading : FCE.txt
--- reading : NLE.txt
--- reading : RPS.txt


In [11]:
portfolios['TRADE_TIME'] = pd.to_datetime(portfolios['TRADE_TIME'])
portfolios['TRADE_DATE'] = pd.to_datetime(portfolios['TRADE_DATE'])

In [12]:
stock_info = pd.read_excel('../../data/stock_data.xlsx')
stock_info = stock_info.drop(['Unnamed: 0'],axis = 1)
stock_info.shape

(282, 4)

In [13]:
stock_info.head()

,symbol,name,buisnesssummary,gics_code
0,HBS,hSenid Business Solutions PLC,An indigenous multinational catering towards m...,45103010 - Application Software
1,TYRE,KELANI TYRES PLC,Manufacturing tyres and tubes and marketing lo...,Automobiles & Components
2,ABL,AMANA BANK PLC,unknown,Banks
3,DFCC,DFCC BANK PLC,The principal activities of DFCC Bank include ...,Banks
4,COMB,COMMERCIAL BANK OF CEYLON PLC,Commercial Banking,Banks


In [14]:
stock_info.shape

(282, 4)

In [15]:
stock_info.symbol.nunique()

282

In [16]:
stock_info[stock_info.buisnesssummary == 'unknown']

,symbol,name,buisnesssummary,gics_code
2,ABL,AMANA BANK PLC,unknown,Banks
11,SDB,SANASA DEVELOPMENT BANK PLC,unknown,Banks
13,UBC,UNION BANK OF COLOMBO PLC,unknown,Banks
15,AEL,ACCESS ENGINEERING PLC,unknown,Capital Goods
33,MEL,MACKWOODS ENERGY PLC,unknown,Capital Goods
...,...,...,...,...
275,HPFL,LOTUS HYDRO POWER PLC,unknown,Utilities
276,PAP,PANASIAN POWER PLC,unknown,Utilities
279,VONE,VALLIBEL ONE PLC,unknown,Utilities
280,LGIL,LOLC GENERAL INSURANCE PLC,unknown,NaN


# Preprocessing

In [17]:
portfolios.shape

(3728894, 8)

In [18]:
portfolios.head(2)

,CDSACCNO,STOCKCODE,REFERANCE,TRAN_TYPE,SHARESQTY,SHAREPRICE,TRADE_DATE,TRADE_TIME
0,BMS-731900310-VN/00,AGPL.N0000,2024153263,B,125,7.5,2024-05-13,2024-05-13 12:25:45
1,BMS-800262640-VN/00,RIL.N0000,2024153264,S,-100,8.5,2024-05-13,2024-05-13 12:25:48


In [19]:
portfolios.TRADE_DATE.min(), portfolios.TRADE_DATE.max()

(Timestamp('2022-01-03 00:00:00'), Timestamp('2024-05-14 00:00:00'))

In [20]:
portfolios.dtypes

CDSACCNO              object
STOCKCODE             object
REFERANCE             object
TRAN_TYPE             object
SHARESQTY              int64
SHAREPRICE           float64
TRADE_DATE    datetime64[ns]
TRADE_TIME    datetime64[ns]
dtype: object

In [21]:
stock_info = stock_info.dropna()

In [22]:
portfolios.shape

(3728894, 8)

In [23]:
print(f"number of unique symbols(items) in portfolios : {portfolios.STOCKCODE.nunique()}")
# print(f"number of unique users in portfolios : {portfolios.CDSACCNO.nunique()}")

number of unique symbols(items) in portfolios : 431


In [24]:
# Only selecting Buy orders
portfolios_df = portfolios.copy()
portfolios_df = portfolios_df.loc[portfolios_df.TRAN_TYPE == 'B']
portfolios_df.shape

(1869633, 8)

In [25]:
portfolios_df['STOCKCODE'] = portfolios_df.STOCKCODE.apply(lambda x : split_symbol(x))

In [26]:
print(f"number of unique symbols(items) in portfolios : {portfolios_df.STOCKCODE.nunique()}")
print(f"number of unique users in portfolios : {portfolios_df.CDSACCNO.nunique()}")

number of unique symbols(items) in portfolios : 288
number of unique users in portfolios : 16173


In [27]:
portfolios_df.shape

(1869633, 8)

In [28]:
#remove symbols that do not have their details in the stock data dataset
unique_port_symbols = set(portfolios_df.STOCKCODE.unique())
unique_symbols = set(stock_info.symbol.unique())

print("unique symbols in portfolio data : {} | unique_symbols in stock details : {}".format(len(unique_port_symbols), len(unique_symbols)))
to_remove = list(unique_port_symbols - unique_symbols)
print("removing {} symbols: {}".format(len(to_remove),to_remove))
portfolios_df = portfolios_df[~portfolios_df.STOCKCODE.isin(to_remove)].reset_index(drop =True)

unique_port_symbols = set(portfolios_df.STOCKCODE.unique())
unique_symbols = set(stock_info.symbol.unique())
print("unique symbols in portfolio data after removing : {} | unique_symbols in stock details : {}".format(len(unique_port_symbols), len(unique_symbols)))

unique symbols in portfolio data : 288 | unique_symbols in stock details : 280
removing 13 symbols: ['LGIL', 'PDL', 'WIND', 'CALI', 'UBF', 'YORK', 'WATA', 'SFL', 'AGPL', 'GSF', 'CLC', 'CBNK', 'CITW']
unique symbols in portfolio data after removing : 275 | unique_symbols in stock details : 280


In [29]:
print(len(portfolios_df))
portfolios_df.tail(2)

1813830


,CDSACCNO,STOCKCODE,REFERANCE,TRAN_TYPE,SHARESQTY,SHAREPRICE,TRADE_DATE,TRADE_TIME
1813828,RPS-760941808-VN/02,PACK,2024005480,B,7,15.1,2024-05-13,2024-05-13 11:18:53
1813829,RPS-797423181-VN/00,LMF,2024005482,B,10,31.8,2024-05-13,2024-05-13 11:48:55


In [30]:
# dropping users with less than k unique items in their portfolio

portfolios_df_fil_1 = portfolios_df.groupby(by = 'CDSACCNO').filter(lambda x: x['STOCKCODE'].nunique() >= min_portfolio_size)
portfolios_df_fil_1['UNIX_TS'] = portfolios_df_fil_1['TRADE_DATE'].apply(lambda x: to_timestamp(x))
portfolios_df_fil_1 = portfolios_df_fil_1.reset_index(drop = True)
portfolios_df_fil_1.head()

,CDSACCNO,STOCKCODE,REFERANCE,TRAN_TYPE,SHARESQTY,SHAREPRICE,TRADE_DATE,TRADE_TIME,UNIX_TS
0,BMS-42281-LI/00,PACK,2024153266,B,300,15.00,2024-05-13,2024-05-13 12:26:03,1.715539e+09
1,BMS-478-LC/00,HNB,2024153267,B,350,202.25,2024-05-13,2024-05-13 12:26:42,1.715539e+09
2,BMS-478-LC/00,HNB,2024153274,B,109,202.25,2024-05-13,2024-05-13 12:29:02,1.715539e+09
3,BMS-522272426-VN/00,ACAP,2024153276,B,350,3.80,2024-05-13,2024-05-13 12:29:24,1.715539e+09
4,BMS-731142378-VN/00,VPEL,2024153282,B,2627,7.90,2024-05-13,2024-05-13 12:30:07,1.715539e+09


In [31]:
portfolios_df_fil_1.CDSACCNO.nunique(), portfolios_df_fil_1.STOCKCODE.nunique()

print("print number of unique users : {}".format(portfolios_df_fil_1.CDSACCNO.nunique()))
print("print number of unique items : {}".format(portfolios_df_fil_1.STOCKCODE.nunique()))

print number of unique users : 2984
print number of unique items : 275


In [32]:
symb_to_name = dict(zip(stock_info.symbol,stock_info.name))
symb_to_gics = dict(zip(stock_info.symbol, stock_info.gics_code))

portfolios_df_fil_1['STOCKNAME'] = portfolios_df_fil_1.STOCKCODE.apply(lambda x: mapper(x,symb_to_name))
portfolios_df_fil_1['GICS'] = portfolios_df_fil_1.STOCKCODE.apply(lambda x: mapper(x,symb_to_gics))

In [33]:
portfolios_df_fil_1[portfolios_df_fil_1.CDSACCNO == "BMS-810392878-VN/00"].shape

(396, 11)

In [34]:
"""
1. calculate ratings for each user and gets max rating for each symbol -> a symbol can only apear once in 
   a user portfolio and the value that appears is the max infered rating for that symbol

2. takes the latest k number of symbols that the user has interacted with -> each user can only have k number of items in the portfolio

3. creates a USER_ID by taking the latest m number of items user has bought as a sequence.
"""

portfolios_df_fil_3 = portfolios_df_fil_1.groupby('CDSACCNO', group_keys= False).apply(lambda x: infer_rating(x)).groupby('CDSACCNO', group_keys= False).apply(lambda x: create_user_portfolios(x, latest_k= max_portfolio_size ,user_id_item_size = user_id_item_size, test_item_size = test_items_size )).reset_index(drop = True) #.sort_values('RATING', ascending= False)

In [35]:
portfolios_df_fil_3.shape

(100349, 14)

In [36]:
portfolios_df_fil_3.test_.value_counts()

test_
0    85429
1    14920
Name: count, dtype: int64

In [37]:
portfolios_df_fil_3.CDSACCNO.value_counts()

CDSACCNO
CAS-551970108-VN/00    50
BMS-522272426-VN/00    50
COM-790042166-VN/00    50
BMS-54087-LI/00        50
BMS-54131-LI/00        50
                       ..
HDF-61833-LI/00        20
BMS-931650912-VN/00    20
HDF-29209-LI/00        20
HDF-760433594-VN/00    20
BMS-523083406-VN/00    20
Name: count, Length: 2984, dtype: int64

In [38]:
portfolios_df_fil_3.STOCKCODE.value_counts()

STOCKCODE
BIL     2185
EXPO    1900
LOFC    1820
LIOC    1813
HAYL    1625
        ... 
HARI       8
SELI       5
SWAD       4
INDO       3
SHAL       2
Name: count, Length: 275, dtype: int64

In [39]:
stockcode_srs = portfolios_df_fil_3.STOCKCODE.value_counts()
stockcode_srs = stockcode_srs[stockcode_srs< minimum_item_occurance]
symbols_to_remove = list(stockcode_srs.index)
symbols_to_remove

['PARA', 'GOOD', 'HARI', 'SELI', 'SWAD', 'INDO', 'SHAL']

In [40]:
portfolios_df_fil_3 = portfolios_df_fil_3[~portfolios_df_fil_3.STOCKCODE.isin(symbols_to_remove)]
portfolios_df_fil_3.CDSACCNO.value_counts()

CDSACCNO
CAS-551970108-VN/00    50
COM-80729-LI/00        50
BMS-53923-LI/00        50
COM-788042957-VN/00    50
COM-790042166-VN/00    50
                       ..
BMS-23226-LI/00        20
HDF-44852-LI/00        20
HDF-66629-LI/00        20
BMS-22958-LI/01        20
CAS-84684-LI/00        20
Name: count, Length: 2984, dtype: int64

In [41]:
portfolios_df_fil_3.CDSACCNO.nunique(), portfolios_df_fil_3.STOCKCODE.nunique()

(2984, 268)

In [42]:
portfolios_df_fil_3.GICS.nunique()

33

In [43]:
# train_ = portfolios_df_fil_3.groupby('CDSACCNO', group_keys=False).apply(lambda x: x.sample(frac=0.8))
# train_ = portfolios_df_fil_3.groupby('CDSACCNO', group_keys=False).apply(lambda x: x.sort_values('UNIX_TS').tail(test_items_size))

# test_ = portfolios_df_fil_4.groupby('CDSACCNO', group_keys=False).apply(lambda x: x.sort_values('UNIX_TS', ascending = True).tail(test_items_size))
test_ = portfolios_df_fil_3[portfolios_df_fil_3.test_ == 1][['USER_ID','CDSACCNO','STOCKCODE','UNIX_TS','RATING','GICS','STOCKNAME']]
test_.head(3)

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
23,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,JINS,1.692643e+09,1.0,Insurance,JANASHAKTHI INSURANCE PLC
24,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,CFLB,1.693247e+09,1.0,Capital Goods,THE COLOMBO FORT LAND AND BUILDING PLC
25,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,CINV,1.693766e+09,1.0,Diversified Financials,CEYLON INVESTMENT PLC


In [44]:
test_.CDSACCNO.value_counts()

CDSACCNO
BMS-10544-LC/00        5
COM-45414-LI/00        5
COM-39680-LI/00        5
COM-40319-LI/00        5
COM-403540153-VN/00    5
                      ..
BMS-73570-LI/00        5
BMS-735822063-VN/00    5
BMS-66201-LI/00        4
BMS-810392878-VN/00    4
BMS-850900701-VN/00    4
Name: count, Length: 2984, dtype: int64

In [45]:
portfolios_df_fil_3[portfolios_df_fil_3.CDSACCNO == "BMS-810392878-VN/00"].shape

(31, 14)

In [46]:
train_ = portfolios_df_fil_3[~portfolios_df_fil_3.index.isin(test_.index)][['USER_ID','CDSACCNO','STOCKCODE','UNIX_TS','RATING','GICS','STOCKNAME']]
# train_

In [47]:
portfolios_df_fil_3 = portfolios_df_fil_3[['USER_ID','CDSACCNO','STOCKCODE','UNIX_TS','RATING','GICS','STOCKNAME']]

In [48]:
train_.sample(frac = 1.0).head(5)

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
53669,"[RCL, AEL, BIL, REEF, COCR, SIRA, HPFL, COMB, ...",BMS-902532811-VN/00,COMB,1.644431e+09,1.0,Banks,COMMERCIAL BANK OF CEYLON PLC
1802,"[SINH, CIC, VPEL, LWL, DIAL, SCAP, AEL, ALUF, ...",BMS-16808-LI/00,AAIC,1.689619e+09,1.0,Insurance,SOFTLOGIC LIFE INSURANCE PLC
64509,"[ASPH, SEMB, AAIC, EXPO, ATL, JKH, CTC, LLUB, ...",CAS-866750998-VN/00,APLA,1.643567e+09,1.0,Materials,ACL PLASTICS PLC
24902,"[SAMP, BIL, SINS, CHOT, PACK, LFIN, MBSL, KZOO...",BMS-651671078-VN/00,JAT,1.679423e+09,1.0,Materials (1510),JAT HOLDINGS PLC
52294,"[LWL, FCT, CFLB, AAIC, RHTL, SCAP, COCR, NTB, ...",BMS-890313701-VN/00,GLAS,1.642617e+09,2.0,Materials,PGP GLASS CEYLON PLC


In [49]:
portfolios_df_fil_3.shape[0] == train_.shape[0] + test_.shape[0]

True

In [50]:
train_.CDSACCNO.nunique(), test_.CDSACCNO.nunique()

(2984, 2984)

In [51]:
train_.CDSACCNO.value_counts()

CDSACCNO
CAS-551970108-VN/00    45
COM-80729-LI/00        45
BMS-53923-LI/00        45
COM-788042957-VN/00    45
COM-790042166-VN/00    45
                       ..
BMS-930980498-VN/00    15
COM-51013-LC/00        15
BMS-26913-LI/00        15
BMS-40691-LI/00        15
BMS-850201684-VN/00    15
Name: count, Length: 2984, dtype: int64

In [52]:
set(train_.STOCKCODE.unique()) - set(test_.STOCKCODE.unique()) , set(test_.STOCKCODE.unique()) - set(train_.STOCKCODE.unique())

({'BLI', 'NEH'}, set())

In [53]:
# train_[train_.CDSACCNO == "BMS-10544-LC/00"].sort_values(by = 'UNIX_TS')

In [54]:
# test_[test_.CDSACCNO == "BMS-10544-LC/00"].sort_values(by = 'UNIX_TS')

In [55]:
len(train_), len(test_)

(85393, 14917)

In [56]:
test_[test_.CDSACCNO == 'BMS-10544-LC/00']

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
23,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,JINS,1.692643e+09,1.0,Insurance,JANASHAKTHI INSURANCE PLC
24,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,CFLB,1.693247e+09,1.0,Capital Goods,THE COLOMBO FORT LAND AND BUILDING PLC
25,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,CINV,1.693766e+09,1.0,Diversified Financials,CEYLON INVESTMENT PLC
26,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,CHOT,1.707935e+09,2.0,Consumer Services,CEYLON HOTELS CORPORATION PLC
27,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,LOFC,1.707935e+09,2.0,Diversified Financials,LOLC FINANCE PLC


In [57]:
train_[train_.CDSACCNO == 'BMS-10544-LC/00']

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
0,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,VONE,1.641148e+09,3.0,Utilities,VALLIBEL ONE PLC
1,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,ELPL,1.641148e+09,3.0,Food Beverage & Tobacco,ELPITIYA PLANTATIONS PLC
2,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,LFIN,1.641407e+09,2.0,Diversified Financials,LB FINANCE PLC
3,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,TKYO,1.641753e+09,2.0,Materials,TOKYO CEMENT COMPANY (LANKA) PLC
4,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,REXP,1.641926e+09,2.0,Materials,RICHARD PIERIS EXPORTS PLC
5,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,KVAL,1.646332e+09,4.0,Food Beverage & Tobacco,KELANI VALLEY PLANTATIONS PLC
6,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,SUN,1.646764e+09,1.0,Food Beverage & Tobacco,SUNSHINE HOLDINGS PLC
7,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,AAIC,1.651171e+09,3.0,Insurance,SOFTLOGIC LIFE INSURANCE PLC
8,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,BIL,1.651603e+09,5.0,Food Beverage & Tobacco,BROWNS INVESTMENTS PLC
9,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,EXPO,1.652035e+09,2.0,Transpotation,EXPOLANKA HOLDINGS PLC


In [58]:
import sys

sys.exit()

SystemExit: 

c:\Users\bpadmin\anaconda3\envs\atrad_cars_v2\lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Saving Data

In [ ]:
data_dict = portfolios_df_fil_3.to_dict(orient='list')
dataset = tf.data.Dataset.from_tensor_slices(data_dict)
save_path = "../../data/portfolios_v2_fixed_max_port_size_{}_useridseq/portfolios".format(max_portfolio_size)
print("portfolios dataset was saved @ {}".format(save_path))
dataset.save(save_path)

portfolios dataset was saved @ ../../data/portfolios_v2_fixed_max_port_size_50_useridseq/portfolios


In [ ]:
data_dict = train_.to_dict(orient='list')
train_dataset = tf.data.Dataset.from_tensor_slices(data_dict)
save_path = "../../data/portfolios_v2_fixed_max_port_size_{}_useridseq/retriver_train".format(max_portfolio_size)
print("train dataset was saved @ {}".format(save_path))
train_dataset.save(save_path)

train dataset was saved @ ../../data/portfolios_v2_fixed_max_port_size_50_useridseq/retriver_train


In [ ]:
data_dict = test_.to_dict(orient='list')
test_dataset = tf.data.Dataset.from_tensor_slices(data_dict)
save_path = "../../data/portfolios_v2_fixed_max_port_size_{}_useridseq/retriver_test".format(max_portfolio_size)
print("test dataset was saved @ {}".format(save_path))
test_dataset.save(save_path)

test dataset was saved @ ../../data/portfolios_v2_fixed_max_port_size_50_useridseq/retriver_test


In [ ]:
len(dataset), len(train_dataset), len(test_dataset)

(100310, 85393, 14917)

In [ ]:
train_hoo, test_hoo = pd.DataFrame(), pd.DataFrame()
for name, group in portfolios_df_fil_3.groupby('CDSACCNO'):
  train_hoo = pd.concat([train_hoo, group.iloc[0:-1]], ignore_index=True)  # Take first row as test
  test_hoo = pd.concat([test_hoo, group.iloc[-1:]], ignore_index=True)  # Rest for train

In [ ]:
train_hoo.CDSACCNO.nunique(), test_hoo.CDSACCNO.nunique()

(2984, 2984)

In [ ]:
train_hoo

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
0,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,VONE,1.641148e+09,3.0,Utilities,VALLIBEL ONE PLC
1,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,ELPL,1.641148e+09,3.0,Food Beverage & Tobacco,ELPITIYA PLANTATIONS PLC
2,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,LFIN,1.641407e+09,2.0,Diversified Financials,LB FINANCE PLC
3,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,TKYO,1.641753e+09,2.0,Materials,TOKYO CEMENT COMPANY (LANKA) PLC
4,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,REXP,1.641926e+09,2.0,Materials,RICHARD PIERIS EXPORTS PLC
...,...,...,...,...,...,...,...
97321,"[SEYB, HNB, AEL, JAT, CALT, ACL, SUN, ALUM, MG...",SCB-1522-LC/00,PLC,1.692124e+09,1.0,Diversified Financials,PEOPLE'S LEASING & FINANCE PLC
97322,"[SEYB, HNB, AEL, JAT, CALT, ACL, SUN, ALUM, MG...",SCB-1522-LC/00,COMB,1.693852e+09,2.0,Banks,COMMERCIAL BANK OF CEYLON PLC
97323,"[SEYB, HNB, AEL, JAT, CALT, ACL, SUN, ALUM, MG...",SCB-1522-LC/00,GLAS,1.702838e+09,1.0,Materials,PGP GLASS CEYLON PLC
97324,"[SEYB, HNB, AEL, JAT, CALT, ACL, SUN, ALUM, MG...",SCB-1522-LC/00,LLUB,1.704220e+09,2.0,Materials,CHEVRON LUBRICANTS LANKA PLC


In [ ]:
test_hoo

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
0,"[COCO, EMER, SAMP, NDB, EML, RIL, SLTL, RCL, C...",BMS-10544-LC/00,LOFC,1.707935e+09,2.0,Diversified Financials,LOLC FINANCE PLC
1,"[LIOC, HELA, AAIC, MELS, NTB, MASK, SLTL, MGT,...",BMS-11214-LC/00,BIL,1.715107e+09,2.0,Food Beverage & Tobacco,BROWNS INVESTMENTS PLC
2,"[BIL, AEL, RCL, HAYL, VONE, TJL, ACL, TKYO, MG...",BMS-11807-LC/00,HELA,1.715539e+09,1.0,Consumer Durables & Apparel,HELA APPAREL HOLDINGS PLC
3,"[SCAP, UBC, RICH, LWL, CFVF, COMB, TILE, CCS, ...",BMS-11829-LI/00,DIAL,1.715193e+09,2.0,Telecommunication Services,DIALOG AXIATA PLC
4,"[EXPO, DIAL, VONE, TKYO, PLR, EDEN, COOP, TESS...",BMS-12282-LI/00,SHOT,1.694371e+09,3.0,Consumer Services,SERENDIB HOTELS PLC
...,...,...,...,...,...,...,...
2979,"[BRWN, EXT, GREG, BFL, HVA, JETS, KFP, VPEL, K...",RPS-953190630-VN/00,WAPO,1.712515e+09,1.0,Diversified Financials,GALLE FACE CAPITAL PARTNERS PLC
2980,"[STAF, TAP, LWL, MGT, SAMP, SHOT, SHL, HNB, HA...",SBK-80957-LC/00,LIOC,1.715279e+09,1.0,Energy,LANKA IOC PLC
2981,"[AEL, JAT, CALT, ACL, SUN, ALUM, MGT, COMB, PL...",SCB-11576-LC/00,CTC,1.713292e+09,2.0,FOOD BEVERAGE & TOBACCO,CEYLON TOBACCO COMPANY PLC
2982,"[HNB, AEL, JAT, CALT, SUN, ALUM, PLC, MGT, COM...",SCB-11577-LC/00,CTC,1.713292e+09,2.0,FOOD BEVERAGE & TOBACCO,CEYLON TOBACCO COMPANY PLC


In [ ]:
data_dict = train_hoo.to_dict(orient='list')
train_hoo_dataset = tf.data.Dataset.from_tensor_slices(data_dict)
train_hoo_dataset.save("../../data/portfolios_v2_fixed_max_port_size_{}_useridseq/retriver_hoo_train".format(max_portfolio_size))

In [ ]:
data_dict = test_hoo.to_dict(orient='list')
test_hoo_dataset = tf.data.Dataset.from_tensor_slices(data_dict)
test_hoo_dataset.save("../../data/portfolios_v2_fixed_max_port_size_{}_useridseq/retriver_hoo_test".format(max_portfolio_size))

In [ ]:
len(dataset), len(train_hoo_dataset), len(test_hoo_dataset)

(100310, 97326, 2984)

In [ ]:
# tf.random.set_seed(42)
# shuffled = dataset.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

# train = shuffled.take(int(len(dataset)* 0.8))
# test = shuffled.skip(int(len(dataset)* 0.8)).take(int(len(dataset)* 0.2))

In [ ]:
# train.save("../../data/portfolios_v2/retriver_train")
# test.save("../../data/portfolios_v2/retriver_test")

In [ ]:
# new_dataset = tf.data.Dataset.load("../../data/portfolios_v2/portfolios_tfds")

In [ ]:
sys.exit()

SystemExit: 

c:\Users\bpadmin\anaconda3\envs\atrad_cars_v2\lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Ranker

In [ ]:
ranker_test_ = portfolios_df_fil_3.groupby('CDSACCNO', group_keys=False).apply(lambda x: x.sort_values('UNIX_TS', ascending = True).tail(10))

In [ ]:
ranker_train_ = portfolios_df_fil_3[~portfolios_df_fil_3.index.isin(ranker_test_.index)][['USER_ID','CDSACCNO','STOCKCODE','UNIX_TS','RATING','GICS','STOCKNAME']]
# train_

In [ ]:
len(ranker_train_), len(ranker_test_), len(ranker_train_)+len(ranker_test_) == len(portfolios_df_fil_3)

(70470, 29840, True)

In [ ]:
ranker_train_data_dict = ranker_train_.to_dict(orient='list')
ranker_train_dataset = tf.data.Dataset.from_tensor_slices(ranker_train_data_dict)

ranker_test_data_dict = ranker_test_.to_dict(orient='list')
ranker_test_dataset = tf.data.Dataset.from_tensor_slices(ranker_test_data_dict)

In [ ]:
import array
import collections

from typing import Dict, List, Optional, Text, Tuple

'''
utility functions to transform normal tensorformat to list wise ranking dataaset
'''
def _create_feature_dict() -> Dict[Text, List[tf.Tensor]]:
  return {"STOCKCODE": [], "RATING": [], "GICS": [], "STOCKNAME": [], "UNIX_TS": []}

def _sample_list(
    feature_lists: Dict[Text, List[tf.Tensor]],
    num_examples_per_list: int,
    random_state: Optional[np.random.RandomState] = None,
) -> Tuple[tf.Tensor, tf.Tensor]:
  """
  Function for sampling a list example from given feature lists.
  """
  
  if random_state is None:
    random_state = np.random.RandomState()

  sampled_indices = random_state.choice(
      range(len(feature_lists["STOCKCODE"])),
      size=num_examples_per_list,
      replace=False,
  )
  sampled_STOCKCODE = [
      feature_lists["STOCKCODE"][idx] for idx in sampled_indices
  ]
  sampled_RATING = [
      feature_lists["RATING"][idx]
      for idx in sampled_indices
  ]
  sampled_GICS = [
      feature_lists["GICS"][idx] for idx in sampled_indices
  ]
  sampled_STOCKNAME = [
      feature_lists["STOCKNAME"][idx]
      for idx in sampled_indices
  ]
  sampled_UNIX_TS = [
      feature_lists["UNIX_TS"][idx] for idx in sampled_indices
  ]

  return (
      tf.stack(sampled_STOCKCODE, 0),
      tf.stack(sampled_RATING, 0),
      tf.stack(sampled_GICS, 0),
      tf.stack(sampled_STOCKNAME, 0),
      tf.stack(sampled_UNIX_TS, 0)
  )


def sample_listwise(
    rating_dataset: tf.data.Dataset,
    num_list_per_user: int = 10,
    num_examples_per_list: int = 10,
    seed: Optional[int] = None,
) -> tf.data.Dataset:
  
  random_state = np.random.RandomState(seed)

  example_lists_by_user = collections.defaultdict(_create_feature_dict)
  cdsaccno_to_userid = dict()

  movie_title_vocab = set()
  for example in rating_dataset:
    user_id = example["CDSACCNO"].numpy()
    user_id_seq = example["USER_ID"].numpy()

    cdsaccno_to_userid[user_id] = user_id_seq

    example_lists_by_user[user_id]["STOCKCODE"].append(
        example["STOCKCODE"])
    example_lists_by_user[user_id]["RATING"].append(
        example["RATING"])
    example_lists_by_user[user_id]["GICS"].append(
        example["GICS"])
    example_lists_by_user[user_id]["STOCKNAME"].append(
        example["STOCKNAME"])
    example_lists_by_user[user_id]["UNIX_TS"].append(
        example["UNIX_TS"])
    
    movie_title_vocab.add(example["STOCKNAME"].numpy())

    

  tensor_slices = {"CDSACCNO": [], "USER_ID" : [], "STOCKCODE": [], "RATING": [], "GICS": [], "STOCKNAME": [], "UNIX_TS": []}

  for user_id, feature_lists in example_lists_by_user.items():
    for _ in range(num_list_per_user):

      # Drop the user if they don't have enough ratings.
      if len(feature_lists["STOCKNAME"]) < num_examples_per_list:
        continue

        '''sampled_STOCKCODE, 0),
      tf.stack(sampled_RATING, 0),
      tf.stack(sampled_GICS, 0),
      tf.stack(sampled_STOCKNAME, 0),
      tf.stack(sampled_UNIX_TS'''

      sampled_STOCKCODE, sampled_RATING, sampled_GICS, sampled_STOCKNAME, sampled_UNIX_TS  = _sample_list(
          feature_lists,
          num_examples_per_list,
          random_state=random_state,
      )
      tensor_slices["CDSACCNO"].append(user_id)
      tensor_slices["USER_ID"].append(cdsaccno_to_userid[user_id])
      tensor_slices["STOCKCODE"].append(sampled_STOCKCODE)
      tensor_slices["RATING"].append(sampled_RATING)
      tensor_slices["GICS"].append(sampled_GICS)
      tensor_slices["STOCKNAME"].append(sampled_STOCKNAME)
      tensor_slices["UNIX_TS"].append(sampled_UNIX_TS)

  return tf.data.Dataset.from_tensor_slices(tensor_slices)

In [ ]:
# portfolios = tf.data.Dataset.load("../../data/portfolios_tfds_lists")
portfolios = dataset

In [ ]:
# train_ds = tf.data.Dataset.load("D:/dev work/recommender systems/Atrad_CARS/data/train_lists").cache() #data\ratings_train
# test_ds = tf.data.Dataset.load("D:/dev work/recommender systems/Atrad_CARS/data/test_lists").cache()

# train_ds = train_dataset
# test_ds = test_dataset

In [ ]:
next(iter(ranker_train_dataset)), len(ranker_train_dataset)

({'USER_ID': <tf.Tensor: shape=(10,), dtype=string, numpy=
  array([b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'RIL', b'SLTL', b'RCL',
         b'CALT', b'PLC'], dtype=object)>,
  'CDSACCNO': <tf.Tensor: shape=(), dtype=string, numpy=b'BMS-10544-LC/00'>,
  'STOCKCODE': <tf.Tensor: shape=(), dtype=string, numpy=b'VONE'>,
  'UNIX_TS': <tf.Tensor: shape=(), dtype=float32, numpy=1641148200.0>,
  'RATING': <tf.Tensor: shape=(), dtype=float32, numpy=3.0>,
  'GICS': <tf.Tensor: shape=(), dtype=string, numpy=b'Utilities'>,
  'STOCKNAME': <tf.Tensor: shape=(), dtype=string, numpy=b'VALLIBEL ONE PLC'>},
 70470)

In [ ]:
train_v1 = sample_listwise(
    ranker_train_dataset,
    num_list_per_user=50,
    num_examples_per_list=10,
    seed=42
)

test_v1 = sample_listwise(
    ranker_test_dataset,
    num_list_per_user=1,
    num_examples_per_list=10,
    seed=42
)

In [ ]:
next(iter(test_v1))

{'CDSACCNO': <tf.Tensor: shape=(), dtype=string, numpy=b'BMS-10544-LC/00'>,
 'USER_ID': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'RIL', b'SLTL', b'RCL',
        b'CALT', b'PLC'], dtype=object)>,
 'STOCKCODE': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'CHOT', b'SLTL', b'JINS', b'RIL', b'CINV', b'RCL', b'LOFC',
        b'PLC', b'CALT', b'CFLB'], dtype=object)>,
 'RATING': <tf.Tensor: shape=(10,), dtype=float32, numpy=array([2., 1., 1., 1., 1., 1., 2., 2., 2., 1.], dtype=float32)>,
 'GICS': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'Consumer Services', b'Telecommunication Services', b'Insurance',
        b'Retailing', b'Diversified Financials', b'Capital Goods',
        b'Diversified Financials', b'Diversified Financials',
        b'Investment Banking & Brokerage', b'Capital Goods'], dtype=object)>,
 'STOCKNAME': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'CEYLON HOTELS CORPORATION PLC', b'SRI L

In [ ]:
next(iter(train_v1))

{'CDSACCNO': <tf.Tensor: shape=(), dtype=string, numpy=b'BMS-10544-LC/00'>,
 'USER_ID': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'RIL', b'SLTL', b'RCL',
        b'CALT', b'PLC'], dtype=object)>,
 'STOCKCODE': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'VONE', b'ELPL', b'BIL', b'KVAL', b'TKYO', b'COCO', b'NDB',
        b'SAMP', b'SPEN', b'LFIN'], dtype=object)>,
 'RATING': <tf.Tensor: shape=(10,), dtype=float32, numpy=array([3., 3., 5., 4., 2., 1., 1., 1., 2., 2.], dtype=float32)>,
 'GICS': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'Utilities', b'Food Beverage & Tobacco',
        b'Food Beverage & Tobacco', b'Food Beverage & Tobacco',
        b'Materials', b'Food Beverage & Tobacco', b'Banks', b'Banks',
        b'Capital Goods', b'Diversified Financials'], dtype=object)>,
 'STOCKNAME': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'VALLIBEL ONE PLC', b'ELPITIYA PLANTATIONS PLC',
        b'BROWNS I

In [ ]:
len(train_v1)

149200

In [ ]:
train_v1.save("../../data/portfolios_v2_fixed_max_port_size_{}_useridseq/ranker_train".format(max_portfolio_size))
test_v1.save("../../data/portfolios_v2_fixed_max_port_size_{}_useridseq/ranker_test".format(max_portfolio_size))

In [ ]:
# train_ds = tf.data.Dataset.load("D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2\retriver_train").cache()